# Steane [[7,1,3]]: one timestep

A single logical qubit, one round of syndrome extraction, with a **synthetic error you
write into the circuit by hand**. The point of the notebook is the one line marked
`ERROR` in section 4 — change it, re-run two cells, read the ancillas.

```
encode |0_L>  ->  YOUR ERROR GATE  ->  syndrome_z ; syndrome_x  ->  read 7 data qubits
```

No decoder, no repeated rounds, no noise model required. Those come later; this is the
piece everything else is built on.

| Section | Content |
| --- | --- |
| 1 | The code: parity checks and the syndrome-to-index map |
| 2 | Encoder for \|0_L> |
| 3 | Syndrome extraction |
| 4 | **The timestep — insert your error here** |
| 5 | Things to try |

## 0. Setup

In [1]:
from collections import Counter

from guppylang import guppy
from guppylang.std.builtins import array, output, py
from guppylang.std.quantum import (
    collect_measurements,
    cx,
    h,
    measure,
    measure_array,
    qubit,
    x,
    y,
    z,
)

from selene_sim import DepolarizingErrorModel, Quest, build

N_DATA = 7
N_QUBITS = 16   # 7 data + 3 ancillas live at a time, plus headroom

## 1. The code

Steane is the [[7,1,3]] CSS code built from the classical [7,4,3] Hamming code. The three
parity checks are chosen so that **column `j` of the check matrix is the binary
representation of `j+1`**:

| qubit `j` | 0 | 1 | 2 | 3 | 4 | 5 | 6 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| address `j+1` | 001 | 010 | 011 | 100 | 101 | 110 | 111 |

Generator `g_k` acts on every qubit whose address has bit `k` set:

```
g0 : {0, 2, 4, 6}      g1 : {1, 2, 5, 6}      g2 : {3, 4, 5, 6}
```

Six stabilizers in total -- `Z` on each support, `X` on each support. The code is
self-dual, so the two families share their supports.

This is what makes the ancillas readable. An `X` error on qubit `j` anticommutes with
exactly those `Z`-stabilizers whose support contains `j`, so the three ancilla bits, read
as a binary number, **are** `j+1`. Zero means no error. No lookup table, no matching.

Changing `CHECKS` is the seam for a different code family; everything below derives
from it.

In [2]:
CHECKS = [[0, 2, 4, 6], [1, 2, 5, 6], [3, 4, 5, 6]]


def syndrome_of(bits):
    """Z-syndrome of a classical bitstring, as an integer. Detects X errors."""
    s = 0
    for k, supp in enumerate(CHECKS):
        if sum(bits[i] for i in supp) % 2:
            s |= 1 << k
    return s


def bits_to_int(three):
    """Three ancilla bits, little-endian, as an integer."""
    return sum(int(b) << k for k, b in enumerate(three))


# The classical oracle every quantum result below is checked against.
for j in range(N_DATA):
    err = [0] * N_DATA
    err[j] = 1
    assert syndrome_of(err) == j + 1, j
print("syndrome_of(X on qubit j) == j+1  for all 7 qubits")

syndrome_of(X on qubit j) == j+1  for all 7 qubits


## 2. Encoding |0_L>

`|0_L>` is the uniform superposition over the eight codewords of the [7,3,4] simplex code,
which is the row space of the check matrix:

```
v0 = 1010101      v1 = 0110011      v2 = 0001111
```

Positions 0, 1 and 3 each carry exactly one of the three vectors, which makes them the
natural seeds: put them in `|+>` to range over all eight combinations, then copy each seed
onto the rest of its support. Three Hadamards, nine CNOTs.

In [3]:
@guppy
def encode_zero() -> array[qubit, 7]:
    """Prepare |0_L> of the Steane code. Ownership of the register passes to the caller."""
    qs = array(qubit() for _ in range(7))
    h(qs[0])
    h(qs[1])
    h(qs[3])
    cx(qs[0], qs[2])   # v0 = 1010101, seeded at 0
    cx(qs[0], qs[4])
    cx(qs[0], qs[6])
    cx(qs[1], qs[2])   # v1 = 0110011, seeded at 1
    cx(qs[1], qs[5])
    cx(qs[1], qs[6])
    cx(qs[3], qs[4])   # v2 = 0001111, seeded at 3
    cx(qs[3], qs[5])
    cx(qs[3], qs[6])
    return qs

### Seeing it

Guppy has no circuit drawer, and the compiled HUGR will not convert to a standalone pytket
circuit — the qubit allocation ops make the region non-contiguous, so `to_tket1()` refuses
it. So the picture is rebuilt from `CHECKS` instead.

That is not a redrawing by hand. The encoder *is* a function of the check matrix: seed
qubits 0, 1 and 3 with a Hadamard, then copy each seed onto the rest of its support. The
cell below derives the gate list from `CHECKS` by that rule, and the `assert` at the end
confirms the result really is `|0_L>` — the eight codewords, uniformly. If you change
`CHECKS`, the drawing follows, and the assert is what catches you if the Guppy encoder
above no longer agrees.

In [4]:
from pytket import Circuit
from pytket.circuit.display import render_circuit_jupyter
from pytket.utils import probs_from_state

SEEDS = [0, 1, 3]   # qubit i carries generator i and nothing else


def encoder_gates():
    """The encoder as a gate list, derived from CHECKS -- same rule as `encode_zero`."""
    gates = [("h", s, s) for s in SEEDS]
    for seed, supp in zip(SEEDS, CHECKS):
        gates += [("cx", seed, t) for t in supp if t != seed]
    return gates


def columns(gates):
    """Greedy layering for drawing: gates share a column only if their spans are disjoint."""
    cols = []
    for kind, a, b in gates:
        lo, hi = min(a, b), max(a, b)
        for col in cols:
            if all(not (min(x, y) <= hi and lo <= max(x, y)) for _, x, y in col):
                col.append((kind, a, b))
                break
        else:
            cols.append([(kind, a, b)])
    return cols


def draw(gates, n=N_DATA):
    """ASCII circuit. Works anywhere, including a plain terminal."""
    rows = [[f"q{i} "] for i in range(n)]
    for col in columns(gates):
        cell = ["───"] * n
        for kind, a, b in col:
            if kind == "h":
                cell[a] = "─H─"
            else:
                cell[a], cell[b] = "─●─", "─⊕─"
                for m in range(min(a, b) + 1, max(a, b)):
                    cell[m] = "─┼─"
        for i in range(n):
            rows[i].append(cell[i])
    return "\n".join("".join(r) + "──" for r in rows)


gates = encoder_gates()
print(draw(gates))

circ = Circuit(N_DATA)
for kind, a, b in gates:
    circ.H(a) if kind == "h" else circ.CX(a, b)

# Does this actually prepare |0_L>? The support of the statevector should be the 8 codewords.
probs = probs_from_state(circ.get_statevector())
support = sorted("".join(map(str, k)) for k, v in probs.items() if v > 1e-9)
assert len(support) == 8, support
assert all(syndrome_of([int(c) for c in w]) == 0 for w in support)

print(f"\n{circ.n_gates} gates ({sum(k == 'h' for k, _, _ in gates)} H, "
      f"{sum(k == 'cx' for k, _, _ in gates)} CX), depth {circ.depth()}")
print("support:", " ".join(support))
print("all 8 are stabilizer-satisfying codewords, so this is |0_L>")

render_circuit_jupyter(circ)

q0 ─H──●──●──●───────────────
q1 ─H──┼──┼──┼──●──●──●──────
q2 ────⊕──┼──┼──⊕──┼──┼──────
q3 ─H──●──┼──┼──●──┼──┼──●───
q4 ────⊕──⊕──┼──┼──┼──┼──┼───
q5 ──────────┼──⊕──⊕──┼──┼───
q6 ──────────⊕────────⊕──⊕───

12 gates (3 H, 9 CX), depth 6
support: 0000000 0001111 0110011 0111100 1010101 1011010 1100110 1101001
all 8 are stabilizer-satisfying codewords, so this is |0_L>


## 3. Syndrome extraction

One bare ancilla per stabilizer, supports written out literally.

- **Z-type** (detects `X` errors): ancilla starts in `|0>`, each data qubit in the support
  acts as control onto it, then measure in `Z`.
- **X-type** (detects `Z` errors): ancilla starts in `|+>`, acts as control onto each data
  qubit in the support, then a Hadamard returns it to the `Z` basis to measure.

The data register is **borrowed** -- `qs: array[qubit, 7]` with no `owned` -- so the caller
keeps it afterwards. Only the ancillas are consumed.

Bare ancillas are not fault-tolerant: a `Z` fault on a Z-type ancilla propagates back
through the CNOTs onto several data qubits at once. That is fine here, where the errors
are the ones you insert deliberately, and it is the term a flagged or Shor-style ancilla
would shrink later.

In [5]:
@guppy
def syndrome_z(qs: array[qubit, 7]) -> array[bool, 3]:
    """Measure the three Z-stabilizers. Detects X errors."""
    a0, a1, a2 = qubit(), qubit(), qubit()

    cx(qs[0], a0)      # g0 : {0, 2, 4, 6}
    cx(qs[2], a0)
    cx(qs[4], a0)
    cx(qs[6], a0)

    cx(qs[1], a1)      # g1 : {1, 2, 5, 6}
    cx(qs[2], a1)
    cx(qs[5], a1)
    cx(qs[6], a1)

    cx(qs[3], a2)      # g2 : {3, 4, 5, 6}
    cx(qs[4], a2)
    cx(qs[5], a2)
    cx(qs[6], a2)

    return array(measure(a0).read(), measure(a1).read(), measure(a2).read())


@guppy
def syndrome_x(qs: array[qubit, 7]) -> array[bool, 3]:
    """Measure the three X-stabilizers. Detects Z errors."""
    a0, a1, a2 = qubit(), qubit(), qubit()
    h(a0)
    h(a1)
    h(a2)

    cx(a0, qs[0])      # g0 : {0, 2, 4, 6}
    cx(a0, qs[2])
    cx(a0, qs[4])
    cx(a0, qs[6])

    cx(a1, qs[1])      # g1 : {1, 2, 5, 6}
    cx(a1, qs[2])
    cx(a1, qs[5])
    cx(a1, qs[6])

    cx(a2, qs[3])      # g2 : {3, 4, 5, 6}
    cx(a2, qs[4])
    cx(a2, qs[5])
    cx(a2, qs[6])

    h(a0)
    h(a1)
    h(a2)
    return array(measure(a0).read(), measure(a1).read(), measure(a2).read())

## 4. The timestep

`report()` builds, runs and tallies whatever program you hand it. Run this cell once.

In [6]:
def report(program, n_shots=100, p=0.0, seed=1, label="timestep"):
    """Build and run a program; tally the (Z-syndrome, X-syndrome) pairs the ancillas gave."""
    kwargs = {}
    if p > 0:
        kwargs["error_model"] = DepolarizingErrorModel(
            p_1q=p, p_2q=p, p_meas=p, p_init=p, random_seed=seed
        )

    runner = build(program.compile())
    tally = Counter()
    for shot in runner.run_shots(
        simulator=Quest(), n_qubits=N_QUBITS, n_shots=n_shots, random_seed=seed, **kwargs
    ):
        rec = dict(shot)
        tally[(bits_to_int(rec["sz"]), bits_to_int(rec["sx"]))] += 1

    print(f"{label}   ({n_shots} shots, p={p})")
    print("  Z-synd  X-synd   shots   the ancillas say")
    for (sz, sx), n in sorted(tally.items(), key=lambda kv: -kv[1]):
        says = []
        if sz:
            says.append(f"X error on q{sz - 1}")
        if sx:
            says.append(f"Z error on q{sx - 1}")
        print(f"  {sz:^6}  {sx:^6}  {n:6d}   {', '.join(says) or 'nothing'}")
    return tally

### Insert the error

This is the cell to edit. The error is an ordinary gate in the circuit, between the
encoder and the extraction:

| want | write |
| --- | --- |
| bit flip on qubit 3 | `x(qs[3])` |
| phase flip on qubit 5 | `z(qs[5])` |
| both on the same qubit | `y(qs[2])` |
| two separate qubits | `x(qs[1])` then `z(qs[6])` |
| nothing | delete the line |

`sz` should read `j+1` for an `X` on qubit `j`; `sx` should read `j+1` for a `Z`. The two
channels are independent, so a `Y` lights up both.

A weight-2 error is where it gets interesting: distance 3 corrects one error, so two will
be misread as a single error somewhere else. Try `x(qs[0])` with `x(qs[1])` and see which
qubit the ancillas blame.

In [7]:
@guppy
def timestep() -> None:
    qs = encode_zero()

    x(qs[3])                                    # <=== ERROR: edit this line

    output("sz", syndrome_z(qs))
    output("sx", syndrome_x(qs))
    output("data", collect_measurements(measure_array(qs)))


report(timestep, n_shots=100, label="x(qs[3])")

x(qs[3])   (100 shots, p=0.0)
  Z-synd  X-synd   shots   the ancillas say
    4       0        100   X error on q3


Counter({(4, 0): 100})

### Sanity check: all fourteen single-qubit Paulis

Same circuit, the error site swept from Python with `py(...)`. Each of the fourteen should
be named exactly, and each should light up one channel only.

Note that `py(j)` bakes `j` in as a constant at compile time, so this is fourteen separate
programs, and `build()` dominates the runtime.

In [8]:
def probe(basis, j):
    """One timestep with a single Pauli of the given basis on qubit j."""
    if basis == "X":

        @guppy
        def prog() -> None:
            qs = encode_zero()
            x(qs[py(j)])
            output("sz", syndrome_z(qs))
            output("sx", syndrome_x(qs))
            output("data", collect_measurements(measure_array(qs)))
    else:

        @guppy
        def prog() -> None:
            qs = encode_zero()
            z(qs[py(j)])
            output("sz", syndrome_z(qs))
            output("sx", syndrome_x(qs))
            output("data", collect_measurements(measure_array(qs)))

    return prog


def syndrome_pair(program, n_shots=20):
    """Run a program and return its single (Z, X) syndrome pair, asserting it is unique."""
    runner = build(program.compile())
    pairs = set()
    for shot in runner.run_shots(
        simulator=Quest(), n_qubits=N_QUBITS, n_shots=n_shots, random_seed=1
    ):
        rec = dict(shot)   # a shot is an iterator: consume it once
        pairs.add((bits_to_int(rec["sz"]), bits_to_int(rec["sx"])))
    assert len(pairs) == 1, f"syndrome not deterministic: {pairs}"
    return pairs.pop()


print("error      Z-synd  X-synd   expected")
for basis in ("X", "Z"):
    for j in range(N_DATA):
        sz, sx = syndrome_pair(probe(basis, j))
        detected = sz if basis == "X" else sx
        other = sx if basis == "X" else sz
        assert detected == j + 1 and other == 0, (basis, j, sz, sx)
        print(f"{basis} on q{j}   {sz:^6}  {sx:^6}   {j + 1}")

print("\nall 14 single-qubit Paulis identified exactly; X and Z channels stay independent")

error      Z-synd  X-synd   expected
X on q0     1       0      1
X on q1     2       0      2
X on q2     3       0      3
X on q3     4       0      4
X on q4     5       0      5
X on q5     6       0      6
X on q6     7       0      7
Z on q0     0       1      1
Z on q1     0       2      2
Z on q2     0       3      3
Z on q3     0       4      4
Z on q4     0       5      5
Z on q5     0       6      6
Z on q6     0       7      7

all 14 single-qubit Paulis identified exactly; X and Z channels stay independent


## 5. Things to try

- **Weight-2 errors.** `x(qs[0])` and `x(qs[1])` together read as `3`, i.e. the decoder
  would blame qubit 2. Distance 3 corrects one error; this is what exceeding it looks
  like from the ancillas' side.
- **A logical operator.** `X_L = X` on all seven qubits. Both syndromes stay `0` -- the
  stabilizers cannot see it, which is precisely what makes it logical.
- **A stabilizer.** `x(qs[0]); x(qs[2]); x(qs[4]); x(qs[6])` is `g0` itself. Also
  invisible, for a different reason: it does not change the state at all.
- **Selene's noise instead of yours.** `report(timestep, p=0.002, n_shots=500)` with the
  error line deleted. Now the syndromes scatter, and the distribution over the eight
  outcomes is the thing to look at.
- **Error after extraction rather than before.** Move the gate below the two `output`
  lines. The ancillas go blank -- a single round only sees what happened before it, which
  is the reason the real experiment repeats rounds and takes differences.